# Demo D — Few labels: how much do labels buy, and where is the structure?

**Class 11 · D1–D2** (slides: `slides/11_autoencoders.md`)

* **D1:** a learning curve on the digits (scikit-learn's 1797 images of 8 × 8 pixels, 64 features, ten classes, the data of Class 05): accuracy of a logistic regression against the number of labelled digits per class (5 seeds). The model is trained **on the labelled rows only** and scored on 540 held-out rows; the other training rows are unlabelled and ignored here.
* **D2:** the unlabelled rows still have a shape. Two views: the digits as images and on a t-SNE map, and the two moons with two labels, with and without the unlabelled points.

In [ ]:
import numpy as np
import polars as pl
from sklearn.datasets import make_moons
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier

from faa import demo, plotlib, semisup
from faa.demo import show, slide

demo.setup("11")
BLUE, ORANGE, GREEN, RED, SKY, PURPLE, GREY = (
    plotlib.BLUE,
    plotlib.ORANGE,
    plotlib.GREEN,
    plotlib.RED,
    plotlib.SKY,
    plotlib.PURPLE,
    plotlib.GREY,
)

## Slide: How Much Do Labels Buy? (D1)

1797 digits: 70 % train (1257 rows), 30 % test (540 rows), stratified. A "label" is the class of one training digit; `per_class` labelled digits of every class are drawn, 5 draws (seeds) for each size.
The model is a logistic regression ($C = 1$) on the 64 pixels scaled to $[0, 1]$.

In [ ]:
X, y = semisup.digits_flat()
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, stratify=y, random_state=0)
rows = []
for per in (1, 2, 3, 5, 10, 20, 50, 110):
    accs = []
    for seed in range(5):
        idx = semisup.few_labels(ytr, per, seed)
        accs.append(LogisticRegression(max_iter=3000).fit(Xtr[idx], ytr[idx]).score(Xte, yte))
    rows.append({"per_class": per, "labels": 10 * per, "acc_mean": np.mean(accs), "acc_std": np.std(accs)})
curve = pl.DataFrame(rows)
full = LogisticRegression(max_iter=3000).fit(Xtr, ytr).score(Xte, yte)
slide("How Much Do Labels Buy?")
print(f"train {len(Xtr)} rows, test {len(Xte)} rows; all {len(Xtr)} labels: accuracy {full:.3f}")
show(curve)

fig, ax = plotlib.fig("col", 1.7)
ax.errorbar(
    curve["labels"],
    curve["acc_mean"],
    curve["acc_std"],
    color=BLUE,
    marker="o",
    ms=3,
    capsize=2,
    label="labelled rows only",
)
ax.axhline(full, color=GREY, ls="--", lw=0.8)
ax.text(12, full - 0.04, f"all labels {full:.3f}", color=GREY, fontsize=6.5)
ax.set_xscale("log")
ax.set_xlabel("labelled digits")
ax.set_ylabel("test accuracy")
ax.legend(loc="lower right")
plotlib.save(fig, "d1_labels_curve")

## Slide: Labels Differ, Not Just Their Number (D1)

With 10 labels (one per class) the accuracy depends on **which** digits were labelled: the spread over 20 draws.

In [ ]:
accs = []
for seed in range(20):
    idx = semisup.few_labels(ytr, 1, seed)
    accs.append(LogisticRegression(max_iter=3000).fit(Xtr[idx], ytr[idx]).score(Xte, yte))
accs = np.array(accs)
slide("Labels Differ, Not Just Their Number")
print(
    f"one label per class, 20 draws: mean {accs.mean():.3f}, std {accs.std():.3f}, min {accs.min():.3f}, max {accs.max():.3f}"
)

fig, ax = plotlib.fig("col", 1.5)
ax.hist(accs, bins=10, color=BLUE, alpha=0.8)
ax.set_xlabel("test accuracy with 10 labels")
ax.set_ylabel("number of draws (of 20)")
plotlib.save(fig, "d1_draws")

## Slide: The Unlabelled Digits Have a Shape (D2)

Left: 40 of the 1257 unlabelled training digits as the 8 × 8 images they are (no label shown). Right: all 1257 on a t-SNE map (a non-linear map used only to draw; Class 09's PCA keeps 28 % of the variance and shows an overlapping blob). The 10 labelled digits (one per class) are marked with their class.
The groups appear without a single label: the **cluster assumption**.

In [ ]:
from matplotlib.gridspec import GridSpec
from sklearn.manifold import TSNE
from sklearn.neighbors import NearestNeighbors

P = TSNE(2, perplexity=30, random_state=0, init="pca").fit_transform(Xtr)
idx = semisup.few_labels(ytr, 1, 0)
pca = PCA(2).fit(Xtr)
fig, ax0 = plotlib.fig("full", 1.85)
ax0.remove()
gs = GridSpec(1, 2, figure=fig, width_ratios=[1, 1.5], wspace=0.08)
axg = fig.add_subplot(gs[0])
rng = np.random.default_rng(0)
pick = rng.choice(len(Xtr), 40, replace=False)
tiles = Xtr[pick].reshape(5, 8, 8, 8).transpose(0, 2, 1, 3).reshape(40, 64)
axg.imshow(1 - tiles, cmap="gray", vmin=0, vmax=1, interpolation="nearest", aspect="equal")
axg.set_xticks([])
axg.set_yticks([])
axg.set_title("40 unlabelled digits", fontsize=7)
ax = fig.add_subplot(gs[1])
ax.scatter(P[:, 0], P[:, 1], s=3, c=GREY, alpha=0.5)
ax.scatter(P[idx, 0], P[idx, 1], s=26, c=ytr[idx], cmap="tab10", edgecolor="k", lw=0.6, zorder=3)
for i in idx:
    ax.annotate(str(ytr[i]), (P[i, 0], P[i, 1]), fontsize=7, xytext=(3, 3), textcoords="offset points", weight="bold")
ax.set_xticks([])
ax.set_yticks([])
ax.set_title("1257 digits on a t-SNE map", fontsize=7)
plotlib.save(fig, "d2_digits_plane")
slide("The Unlabelled Digits Have a Shape")
print(
    f"PCA keeps {100 * pca.explained_variance_ratio_.sum():.1f} % of the variance in 2 components; t-SNE is a non-linear map that keeps neighbours together"
)
# how well do the groups match the hidden classes? neighbour purity of the map
nb = NearestNeighbors(n_neighbors=6).fit(P).kneighbors(P, return_distance=False)[:, 1:]
print(
    f"on the map, {np.mean(ytr[nb] == ytr[:, None]):.3f} of the 5 nearest neighbours of a digit share its (hidden) class"
)

## Slide: Two Labels and the Cluster Assumption (D2)

Two moons (300 points, noise 0.08), one labelled point per class, **both models see only the 2 labelled points**.
A line (logistic regression) and 1-nearest neighbour give the same boundary (the perpendicular bisector of the two points), hence the same accuracy: neither uses the 298 unlabelled points.
Right: label spreading (Demo F) uses them through a nearest-neighbour graph.

In [ ]:
Xm, ym = make_moons(300, noise=0.08, random_state=1)
lab = np.array([np.argmax((ym == 0) & (Xm[:, 0] < -0.8)), np.argmax((ym == 1) & (Xm[:, 0] > 1.8))])
lr = LogisticRegression().fit(Xm[lab], ym[lab])
nn = KNeighborsClassifier(1).fit(Xm[lab], ym[lab])
Wm = semisup.knn_affinity(Xm, 10)
sp = semisup.label_spreading(Wm, ym, lab, 0.99)
slide("Two Labels and the Cluster Assumption")
print(
    f"labelled rows {lab.tolist()} (classes {ym[lab].tolist()}); logistic regression accuracy {lr.score(Xm, ym):.3f}; 1-NN {nn.score(Xm, ym):.3f} (same boundary); label spreading, which also uses the 298 unlabelled points: {np.mean(sp == ym):.3f}"
)
print(f"the two predictions agree on {np.mean(lr.predict(Xm) == nn.predict(Xm)):.3f} of the points")

fig, ax0 = plotlib.fig("full", 1.35)
ax0.remove()
axs = fig.subplots(1, 2, sharey=True)
gx, gy = np.meshgrid(np.linspace(-1.5, 2.5, 200), np.linspace(-1, 1.5, 200))
zz = lr.predict(np.c_[gx.ravel(), gy.ravel()]).reshape(gx.shape)
axs[0].contourf(gx, gy, zz, levels=[-0.5, 0.5, 1.5], colors=[SKY, "#F5D9A8"], alpha=0.5)
axs[0].scatter(Xm[:, 0], Xm[:, 1], s=4, c=np.where(ym == 0, BLUE, ORANGE), alpha=0.7)
axs[0].set_title("2 labels only: line = 1-NN", fontsize=7)
axs[1].scatter(Xm[:, 0], Xm[:, 1], s=4, c=np.where(sp == 0, BLUE, ORANGE), alpha=0.7)
axs[1].set_title("2 labels + 298 unlabelled", fontsize=7)
for a_ in axs:
    a_.scatter(Xm[lab, 0], Xm[lab, 1], s=40, c=[BLUE, ORANGE], edgecolor="k", zorder=3)
    a_.set_xlabel("$x_1$")
axs[0].set_ylabel("$x_2$")
plotlib.save(fig, "d2_moons_line")

## Failure case: a bad draw

The curve hides the worst draw: print the smallest accuracy for each size.

In [ ]:
rows = []
for per in (1, 5, 20):
    accs = []
    for seed in range(20):
        idx = semisup.few_labels(ytr, per, seed)
        accs.append(LogisticRegression(max_iter=3000).fit(Xtr[idx], ytr[idx]).score(Xte, yte))
    rows.append({"per_class": per, "worst": min(accs), "median": float(np.median(accs)), "best": max(accs)})
slide("Labels Differ, Not Just Their Number (worst draw)")
show(pl.DataFrame(rows))